In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from typing import Any

from project_root import PROJECT_ROOT,DATASETS_ROOT

import fiftyone as fo
import fiftyone.utils.torch as fout


import torchvision as tv

from scripts.model_serialization import load_model

no_grad_guard = torch.no_grad()

2025-04-04 12:04:50.228494: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-04 12:04:50.235413: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743761090.243303  699170 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743761090.245690  699170 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743761090.251851  699170 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [6]:
# dataset_name =         "zoo-elephants-identity-tracks"
# ds =fo.load_dataset(dataset_name)


def make_ds():
    return fo.Dataset.from_dir(
        dataset_type=fo.types.ImageDirectory,
        dataset_dir="/media/dherrera/ElephantExternal/elephants/tracks/tracks_apr03/bad",
        persistent=False,
    )


ds = make_ds()
# classes = sorted(ds.classes["ground_truth"])
# print(classes)
print(ds)

 100% |███████████████| 1372/1372 [87.6ms elapsed, 0s remaining, 15.7K samples/s]  
Name:        2025.04.04.12.06.24
Media type:  image
Num samples: 1372
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField


In [ ]:
# Compute uniqueness
import fiftyone.brain as fob

session = fo.launch_app(ds, auto=False)
session.open_tab()

Session launched. Run `session.show()` to open the App in a cell output.


<IPython.core.display.Javascript object>

In [ ]:
from tqdm.auto import tqdm

dup_result = fob.compute_near_duplicates(ds)
print(len(dup_result.duplicate_ids))
session.view = dup_result.duplicates_view()

DELETE_DUPLICATES = True
if DELETE_DUPLICATES:
    while len(dup_result.duplicate_ids) > 0:
        print(f"Deleting {len(dup_result.duplicate_ids)} duplicate files")
        for id in tqdm(dup_result.duplicate_ids):
            filepath = ds[id]["filepath"]
            Path.unlink(filepath)
        ds = make_ds()
        dup_result = fob.compute_near_duplicates(ds)
        print(len(dup_result.duplicate_ids))

Computing embeddings...
 100% |███████████████| 1830/1830 [4.6s elapsed, 0s remaining, 403.3 samples/s]       
Computing duplicate samples...
Generating index for 1830 embeddings...
Index complete
Generating neighbors graph for 1712 embeddings...
Index complete
Duplicates computation complete
118
Deleting 118 duplicate files


  0%|          | 0/118 [00:00<?, ?it/s]

 100% |███████████████| 1712/1712 [104.6ms elapsed, 0s remaining, 16.4K samples/s] 
Computing embeddings...
 100% |███████████████| 1712/1712 [4.7s elapsed, 0s remaining, 313.1 samples/s]       
Computing duplicate samples...
Generating index for 1712 embeddings...
Index complete
Generating neighbors graph for 1686 embeddings...
Index complete
Duplicates computation complete
26
Deleting 26 duplicate files


  0%|          | 0/26 [00:00<?, ?it/s]

 100% |███████████████| 1686/1686 [113.3ms elapsed, 0s remaining, 15.0K samples/s] 
Computing embeddings...
 100% |███████████████| 1686/1686 [5.3s elapsed, 0s remaining, 309.4 samples/s]       
Computing duplicate samples...
Generating index for 1686 embeddings...
Index complete
Generating neighbors graph for 1681 embeddings...
Index complete
Duplicates computation complete
5
Deleting 5 duplicate files


  0%|          | 0/5 [00:00<?, ?it/s]

 100% |███████████████| 1681/1681 [113.9ms elapsed, 0s remaining, 14.9K samples/s] 
Computing embeddings...
 100% |███████████████| 1681/1681 [5.4s elapsed, 0s remaining, 317.4 samples/s]       
Computing duplicate samples...
Generating index for 1681 embeddings...
Index complete
Generating neighbors graph for 1680 embeddings...
Index complete
Duplicates computation complete
1
Deleting 1 duplicate files


  0%|          | 0/1 [00:00<?, ?it/s]

 100% |███████████████| 1680/1680 [114.7ms elapsed, 0s remaining, 14.9K samples/s]  
Computing embeddings...
 100% |███████████████| 1680/1680 [5.3s elapsed, 0s remaining, 359.2 samples/s]       
Computing duplicate samples...
Generating index for 1680 embeddings...
Index complete
Duplicates computation complete
0


In [ ]:
from fiftyone import ViewField as F

THRESHOLD = 0.01
fob.compute_uniqueness(ds)
session.view = ds.sort_by("uniqueness")

ds_bad = ds.match(F("uniqueness") < THRESHOLD)
print(f"Samples to drop: {len(ds_bad)}")
session.view = ds_bad

from tqdm.auto import tqdm

DELETE_BAD = True
if DELETE_BAD:
    while len(ds_bad) > 0:
        print(f"Deleting {len(ds_bad)} low-uniqueness files")
        for sample in tqdm(ds_bad):
            filepath = sample["filepath"]
            Path.unlink(filepath)
        ds = make_ds()
        fob.compute_uniqueness(ds)
        ds_bad = ds.match(F("uniqueness") < THRESHOLD)

session.view = ds

Computing embeddings...
 100% |███████████████| 1680/1680 [682.5ms elapsed, 0s remaining, 2.5K samples/s]      
Computing uniqueness...
Generating index for 1680 embeddings...
Index complete
Uniqueness computation complete
Samples to drop: 264
Deleting 264 duplicate files


  0%|          | 0/264 [00:00<?, ?it/s]

 100% |███████████████| 1416/1416 [101.1ms elapsed, 0s remaining, 14.0K samples/s] 
Computing embeddings...
 100% |███████████████| 1416/1416 [680.9ms elapsed, 0s remaining, 2.1K samples/s]      
Computing uniqueness...
Generating index for 1416 embeddings...
Index complete
Uniqueness computation complete
Deleting 6 duplicate files


  0%|          | 0/6 [00:00<?, ?it/s]

 100% |███████████████| 1410/1410 [127.6ms elapsed, 0s remaining, 11.2K samples/s]  
Computing embeddings...
 100% |███████████████| 1410/1410 [673.8ms elapsed, 0s remaining, 2.1K samples/s]      
Computing uniqueness...
Generating index for 1410 embeddings...
Index complete
Uniqueness computation complete
Deleting 38 duplicate files


  0%|          | 0/38 [00:00<?, ?it/s]

 100% |███████████████| 1372/1372 [88.9ms elapsed, 0s remaining, 15.4K samples/s]  
Computing embeddings...
 100% |███████████████| 1372/1372 [620.2ms elapsed, 0s remaining, 2.2K samples/s]      
Computing uniqueness...
Generating index for 1372 embeddings...
Index complete
Uniqueness computation complete


ValueError: `Session.view` must be a <class 'fiftyone.core.view.DatasetView'> or None; found <class 'fiftyone.core.dataset.Dataset'>